# ASSIGNMENT 01:22I1865- Aqsa Fayaz

Comparative Multi-Task Affect Recognition (ResNet vs. EfficientNet) :  a comparative study using a multi-task CNN architecture with two popular backbones: ResNet50 and EfficientNetB0. The goal is to simultaneously predict categorical Expression (8 classes) and continuous Valence/Arousal values (regression), adhering to all specified metrics and data handling requirements.



# Environment Setup and Custom Metrics 
We'll define the required custom metrics, especially the Concordance Correlation Coefficient (CCC), which is essential for the continuous affective domain.

In [11]:
# Install necessary libraries if not already present
# !pip install tensorflow numpy scikit-learn

import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence, to_categorical
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, average_precision_score
from tensorflow.keras import backend as K

# Set a random seed for reproducibility
tf.random.set_seed(42)

# --- Custom Metrics ---

def ccc_metric(y_true, y_pred):
    """
    Concordance Correlation Coefficient (CCC) Keras metric.
    Measures agreement between true and predicted continuous values.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    mean_true = K.mean(y_true, axis=0)
    mean_pred = K.mean(y_pred, axis=0)
    
    var_true = K.var(y_true, axis=0)
    var_pred = K.var(y_pred, axis=0)
    
    std_true = K.std(y_true, axis=0)
    std_pred = K.std(y_pred, axis=0)
    
    # Covariance
    cov = K.mean((y_true - mean_true) * (y_pred - mean_pred), axis=0)
    
    # CCC formula
    numerator = 2.0 * cov
    denominator = var_true + var_pred + K.square(mean_true - mean_pred)
    
    return K.mean(numerator / (denominator + K.epsilon())) # Added K.epsilon for stability

def sagr_metric(y_true, y_pred):
    """
    Sign Agreement Metric (SAGR) Keras metric.
    Measures the percentage of samples where the predicted sign matches the true sign.
    """
    y_true_sign = K.sign(y_true)
    y_pred_sign = K.sign(y_pred)
    
    agreement = K.equal(y_true_sign, y_pred_sign)
    
    return K.mean(K.cast(agreement, 'float32'))

# 2. Data Preparation and Generator (Refined) 
This section refines the data loading process to:

Collect all file IDs and their corresponding labels.

Filter out samples where Valence or Arousal is the invalid value of -2 (for 'Uncertain' and 'No-face' categories), ensuring the regression tasks are trained only on valid [−1,+1] data.

Implement the efficient Keras Sequence generator.

In [2]:
import os

# --- Configuration ---
# Set DATA_DIR to the root dataset folder
DATA_DIR = r'E:\Semester 7\Deep Learning\22i1865_Assignment_01\Dataset'

# Update paths for images and annotations
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
ANNOTATIONS_DIR = os.path.join(DATA_DIR, 'annotations')
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8
INVALID_LABEL = -2.0 # Valence/Arousal value for 'Uncertain' or 'No-face'

# --- 1. Master List Creation and Filtering ---
def create_master_list(images_dir, annotations_dir):
    """
    Gathers all file IDs and filters out samples with invalid Valence/Arousal (-2).
    """
    all_files = glob.glob(os.path.join(images_dir, '*.jpg'))
    master_data = []

    for img_path in all_files:
        base_id = os.path.splitext(os.path.basename(img_path))[0]
        
        # Load labels
        try:
            val = np.load(os.path.join(annotations_dir, f'{base_id}_val.npy')).item()
            aro = np.load(os.path.join(annotations_dir, f'{base_id}_aro.npy')).item()
            exp = np.load(os.path.join(annotations_dir, f'{base_id}_exp.npy')).item()

            # Filter out invalid regression targets (Valence/Arousal = -2)
            if val != INVALID_LABEL and aro != INVALID_LABEL:
                master_data.append({
                    'id': base_id, 
                    'exp': exp, 
                    'val': val, 
                    'aro': aro
                })
        except FileNotFoundError:
            # Skip if a label file is missing
            continue
            
    return pd.DataFrame(master_data)

# --- 2. Keras Data Generator ---

class AffectDataGenerator(Sequence):
    """Keras Sequence for efficient multi-task data loading."""
    def __init__(self, data_df, images_dir, batch_size, image_size, n_classes, shuffle=True):
        self.data_df = data_df.reset_index(drop=True)  # Reset index to avoid issues
        self.images_dir = images_dir
        self.batch_size = batch_size
        self.image_size = image_size
        self.n_classes = n_classes
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.data_df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.data_df = self.data_df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, index):
        batch_start = index * self.batch_size
        batch_end = min((index + 1) * self.batch_size, len(self.data_df))  # Handle last batch
        batch_df = self.data_df.iloc[batch_start:batch_end].reset_index(drop=True)
        
        actual_batch_size = len(batch_df)  # Get actual batch size for this batch
        
        X = np.empty((actual_batch_size, *self.image_size, 3))
        y_exp = np.empty((actual_batch_size, self.n_classes), dtype=int)
        y_val = np.empty((actual_batch_size, 1), dtype=np.float32)
        y_aro = np.empty((actual_batch_size, 1), dtype=np.float32)

        for i, (idx, row) in enumerate(batch_df.iterrows()):
            try:
                # Image Loading (Input) - Fixed path issue with debugging
                image_path = os.path.join(self.images_dir, f'{row["id"]}.jpg')
                
                # Debug: Check if file exists
                if not os.path.exists(image_path):
                    print(f"File not found: {image_path}")
                    # Fill with zeros for missing files
                    X[i,] = np.zeros((*self.image_size, 3))
                    y_exp[i,] = np.zeros(self.n_classes)
                    y_val[i,] = 0.0
                    y_aro[i,] = 0.0
                    continue
                    
                img = load_img(image_path, target_size=self.image_size)
                X[i,] = img_to_array(img) / 255.0

                # Label Assignment (Outputs)
                y_exp[i,] = to_categorical(row['exp'], num_classes=self.n_classes)
                y_val[i,] = row['val']
                y_aro[i,] = row['aro']
                
            except Exception as e:
                print(f"Error processing {row['id']}: {e}")
                # Fill with zeros for failed samples
                X[i,] = np.zeros((*self.image_size, 3))
                y_exp[i,] = np.zeros(self.n_classes)
                y_val[i,] = 0.0
                y_aro[i,] = 0.0
            
        return X, {'expression_output': y_exp, 'valence_output': y_val, 'arousal_output': y_aro}

# --- Execution of Data Loading ---
full_data_df = create_master_list(IMAGES_DIR, ANNOTATIONS_DIR)

if full_data_df.empty:
    raise FileNotFoundError("No valid image/label pairs found. Check DATA_DIR and file structure.")

# Split the DataFrame
train_df, val_df = train_test_split(full_data_df, test_size=0.2, random_state=42)

# Instantiate the generators - Fixed to use IMAGES_DIR
train_generator = AffectDataGenerator(train_df, IMAGES_DIR, BATCH_SIZE, IMAGE_SIZE, NUM_CLASSES, shuffle=True)
validation_generator = AffectDataGenerator(val_df, IMAGES_DIR, BATCH_SIZE, IMAGE_SIZE, NUM_CLASSES, shuffle=False)

print(f"Total valid samples (Val/Aro in [-1, +1]): {len(full_data_df)}")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Training batches: {len(train_generator)}")
print(f"Validation batches: {len(validation_generator)}")

# Test the generator to make sure it works
print("Testing data generator...")
try:
    sample_batch = train_generator[0]
    print("Data generator test successful!")
    print(f"Batch shape - Images: {sample_batch[0].shape}")
    print(f"Batch shape - Expression: {sample_batch[1]['expression_output'].shape}")
    print(f"Batch shape - Valence: {sample_batch[1]['valence_output'].shape}")
    print(f"Batch shape - Arousal: {sample_batch[1]['arousal_output'].shape}")
    
    # Test last batch to ensure it handles variable batch sizes
    last_batch_idx = len(train_generator) - 1
    if last_batch_idx > 0:
        last_batch = train_generator[last_batch_idx]
        print(f"Last batch shape - Images: {last_batch[0].shape}")
        
except Exception as e:
    print(f"Data generator test failed: {e}")

Total valid samples (Val/Aro in [-1, +1]): 3999
Training samples: 3199
Validation samples: 800
Training batches: 99
Validation batches: 25
Testing data generator...
Data generator test successful!
Batch shape - Images: (32, 224, 224, 3)
Batch shape - Expression: (32, 8)
Batch shape - Valence: (32, 1)
Batch shape - Arousal: (32, 1)
Last batch shape - Images: (32, 224, 224, 3)
Last batch shape - Images: (32, 224, 224, 3)


# 3. Comparative Multi-Task Model Builder 
This function abstracts the architecture, allowing us to easily switch between ResNet50 and EfficientNetB0.

In [15]:
def build_multi_task_cnn(input_shape, num_classes, base_model_name):
    """
    Builds the multi-task CNN using a specified pre-trained backbone.
    """
    input_tensor = Input(shape=input_shape, name='input_image')
    
    if base_model_name == 'ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=input_tensor)
    elif base_model_name == 'EfficientNetB0':
        base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_tensor)
    else:
        raise ValueError("Invalid base_model_name. Use 'ResNet50' or 'EfficientNetB0'.")

    # Freeze the base layers for transfer learning
    base_model.trainable = False
        
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Branch 1: Expression Classification (Softmax)
    exp_head = Dense(512, activation='relu')(x)
    expression_output = Dense(num_classes, activation='softmax', name='expression_output')(exp_head)
    
    # Branch 2: Valence Regression (Linear)
    val_head = Dense(256, activation='relu')(x)
    valence_output = Dense(1, activation='linear', name='valence_output')(val_head)
    
    # Branch 3: Arousal Regression (Linear)
    aro_head = Dense(256, activation='relu')(x)
    arousal_output = Dense(1, activation='linear', name='arousal_output')(aro_head)
    
    model = Model(
        inputs=base_model.input, 
        outputs=[expression_output, valence_output, arousal_output],
        name=f'{base_model_name}_MultiTask_Model'
    )
    
    return model

# --- Compilation and Training Setup (Re-usable function) ---
def train_and_evaluate(model, train_gen, val_gen, model_name, epochs=30):
    
    print(f"\n{'='*60}\nStarting Training for: {model_name}\n{'='*60}")
    
    loss_functions = {
        'expression_output': 'categorical_crossentropy', 
        'valence_output': 'mse',
        'arousal_output': 'mse'
    }

    metrics = {
        'expression_output': ['accuracy'], # F1, AUC, Kappa calculated manually later
        'valence_output': ['mse', ccc_metric, sagr_metric],
        'arousal_output': ['mse', ccc_metric, sagr_metric]
    }
    
    # Loss Weights (Crucial for multi-task balance)
    loss_weights = {
        'expression_output': 1.0, 
        'valence_output': 1.5,     # Increased weight for regression
        'arousal_output': 1.5
    }

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=loss_functions,
        metrics=metrics,
        loss_weights=loss_weights
    )

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'{model_name}_best.keras',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        ),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, verbose=1)
    ]

    history = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=callbacks,
        verbose=1
    )
    
    # Load and return the best model for final evaluation
    try:
        best_model = tf.keras.models.load_model(
            f'{model_name}_best.keras', 
            custom_objects={'ccc_metric': ccc_metric, 'sagr_metric': sagr_metric}
        )
    except:
        best_model = model # Fallback if saving failed
        
    return best_model, history

# 4. Comparative Training and Evaluation 
We now run the training process for both baselines and consolidate the results.

4.1. Run ResNet50 Baseline
Python

In [19]:
resnet_model = build_multi_task_cnn(IMAGE_SIZE + (3,), NUM_CLASSES, 'ResNet50')
resnet_best_model, resnet_history = train_and_evaluate(
    resnet_model, train_generator, validation_generator, 'ResNet50', epochs=30
)

# Evaluate and interpret the ResNet50 performance with detailed loss analysis
resnet_metrics = evaluate_model_performance_with_losses(resnet_history, 'ResNet50')


Starting Training for: ResNet50
Epoch 1/30
Epoch 1/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - arousal_output_ccc_metric: 0.0048 - arousal_output_loss: 0.1667 - arousal_output_mse: 0.1667 - arousal_output_sagr_metric: 0.7579 - expression_output_accuracy: 0.1282 - expression_output_loss: 2.1038 - loss: 2.6973 - valence_output_ccc_metric: 0.0015 - valence_output_loss: 0.2289 - valence_output_mse: 0.2289 - valence_output_sagr_metric: 0.6746 - val_arousal_output_ccc_metric: 0.0034 - val_arousal_output_loss: 0.1478 - val_arousal_output_mse: 0.1478 - val_arousal_output_sagr_metric: 0.7800 - val_expression_output_accuracy: 0.1075 - val_expression_output_loss: 2.0904 - val_loss: 2.6772 - val_valence_output_ccc_metric: 0.0023 - val_valence_output_loss: 0.2435 - val_valence_output_mse: 0.2435 - val_valence_output_sagr_metric: 0.7013
Epoch 2/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - arousal_output_ccc_metric: 0.0048 - arousal_output_loss: 0.1667 - arousal_output_mse: 0.1667 - arousal_output_

In [10]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- Required Custom Metrics Definitions ---
# These must be defined again for both model loading and recompilation.
def ccc_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    mean_true = K.mean(y_true, axis=0)
    mean_pred = K.mean(y_pred, axis=0)
    var_true = K.var(y_true, axis=0)
    var_pred = K.var(y_pred, axis=0)
    cov = K.mean((y_true - mean_true) * (y_pred - mean_pred), axis=0)
    
    numerator = 2.0 * cov
    denominator = var_true + var_pred + K.square(mean_true - mean_pred)
    
    return K.mean(numerator / (denominator + K.epsilon()))

def sagr_metric(y_true, y_pred):
    y_true_sign = K.sign(y_true)
    y_pred_sign = K.sign(y_pred)
    agreement = K.equal(y_true_sign, y_pred_sign)
    return K.mean(K.cast(agreement, 'float32'))


# --- 1. Load the Best Model from the Frozen Phase ---
print("Loading best model weights from the initial frozen phase...")
best_resnet_model = tf.keras.models.load_model(
    'ResNet50_best.keras', 
    custom_objects={'ccc_metric': ccc_metric, 'sagr_metric': sagr_metric}
)

# --- 2. Unfreeze the Entire Base Model for Fine-Tuning ---
print("Unfreezing ResNet50 base layers for fine-tuning...")
for layer in best_resnet_model.layers:
    # Set all layers to trainable except the Input layer
    if 'input' not in layer.name:
        layer.trainable = True

# --- 3. Define New Training Parameters ---
FINE_TUNING_LR = 1e-5 

fine_tuning_loss_weights = {
    'expression_output': 1.0,  
    'valence_output': 2.0,     # Increased weight for continuous task
    'arousal_output': 2.5      # Increased weight for continuous task
}

# --- 4. Recompile the Model (FIX APPLIED HERE) ---
# We explicitly define the metrics dictionary for the three outputs.
metrics_for_recompile = {
    'expression_output': ['accuracy'],
    'valence_output': ['mse', ccc_metric, sagr_metric],
    'arousal_output': ['mse', ccc_metric, sagr_metric]
}

print("Recompiling model with low learning rate and adjusted loss weights...")

best_resnet_model.compile(
    optimizer=Adam(learning_rate=FINE_TUNING_LR),
    loss=best_resnet_model.loss,
    metrics=metrics_for_recompile, # <-- Using the explicit dictionary to fix the ValueError
    loss_weights=fine_tuning_loss_weights
)

# --- 5. Continue Training (Fine-Tuning Phase) ---
print("\nStarting RESNET Fine-Tuning Phase (Lower LR, Unfrozen Layers)...")

fine_tune_callbacks = [
    ModelCheckpoint(
        filepath='ResNet50_fine_tuned_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1) 
]

# NOTE: You must ensure 'train_generator' and 'validation_generator' objects are available in your environment.
history_fine_tune = best_resnet_model.fit(
    train_generator,
    epochs=50, 
    validation_data=validation_generator,
    callbacks=fine_tune_callbacks,
    verbose=1
)

Loading best model weights from the initial frozen phase...
Unfreezing ResNet50 base layers for fine-tuning...
Recompiling model with low learning rate and adjusted loss weights...

Starting RESNET Fine-Tuning Phase (Lower LR, Unfrozen Layers)...
Unfreezing ResNet50 base layers for fine-tuning...
Recompiling model with low learning rate and adjusted loss weights...

Starting RESNET Fine-Tuning Phase (Lower LR, Unfrozen Layers)...


c:\Users\ahmed\miniconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


KeyboardInterrupt: 

In [22]:
# --- PHASE 1: RESNET ULTRA-FINE-TUNING (FIXING OVERFITTING) ---
print("--- RESNET PHASE 1: ULTRA-FINE-TUNING (FIXING OVERFITTING) ---")

# 1. Load the Best Model from the Overfitting Phase
print("Loading best model weights from the current run...")
best_overfitted_resnet_model = tf.keras.models.load_model(
    'ResNet50_fine_tuned_best.keras', 
    custom_objects={'ccc_metric': ccc_metric, 'sagr_metric': sagr_metric}
)

# 2. Ultra-Low Learning Rate (CRUCIAL FIX)
ULTRA_FINE_TUNING_LR = 1e-6 

print(f"Recompiling model with Ultra-Low Learning Rate: {ULTRA_FINE_TUNING_LR}")
best_overfitted_resnet_model.compile(
    optimizer=Adam(learning_rate=ULTRA_FINE_TUNING_LR), 
    loss=best_overfitted_resnet_model.loss,
    metrics=metrics_for_recompile,
    loss_weights=fine_tuning_loss_weights
)

# 3. Continue Training 
print("\nStarting RESNET Ultra-Fine-Tuning Phase...")
ultra_fine_tune_callbacks = [
    ModelCheckpoint(
        filepath='ResNet50_FINAL_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=15, verbose=1) # Increased patience for slow learning
]

# NOTE: Requires 'train_generator' and 'validation_generator'
history_ultra_fine_tune = best_overfitted_resnet_model.fit(
    train_generator,
    epochs=50, 
    validation_data=validation_generator,
    callbacks=ultra_fine_tune_callbacks,
    verbose=1
)
print("ResNet Ultra-Fine-Tuning Complete. File saved: ResNet50_FINAL_best.keras")


# --- PHASE 2: EFFICIENTNET COMBINED TRAINING ---
print("\n" + "="*80)
print("--- EFFICIENTNET PHASE 2: COMBINED TRAINING ---")
print("Unfrozen base, fast-track training due to deadline.")
print("="*80)

# 1. Build and Unfreeze Immediately
# NOTE: Requires 'build_multi_task_cnn', 'IMAGE_SIZE', and 'NUM_CLASSES'
efficientnet_model = build_multi_task_cnn(IMAGE_SIZE + (3,), NUM_CLASSES, 'EfficientNetB0')
efficientnet_model.trainable = True

EFFICIENTNET_LR = 5e-6 # Slightly faster LR for a fresh, more efficient model

# 2. Compile
efficientnet_model.compile(
    optimizer=Adam(learning_rate=EFFICIENTNET_LR),
    loss=efficientnet_model.loss,
    metrics=metrics_for_recompile, 
    loss_weights=fine_tuning_loss_weights 
)

# 3. Train
efficientnet_callbacks = [
    ModelCheckpoint(
        filepath='EfficientNet_FINAL_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1) 
]

history_efficientnet = efficientnet_model.fit(
    train_generator,
    epochs=50, 
    validation_data=validation_generator,
    callbacks=efficientnet_callbacks,
    verbose=1
)
print("EfficientNet Combined Training Complete. File saved: EfficientNet_FINAL_best.keras")


# --- PHASE 3: FINAL METRIC COMPARISON ---
print("\n" + "="*80)
print("--- PHASE 3: FINAL METRIC COMPARISON ---")
print("Generating final submission table...")
print("="*80)

# 1. Load the final, best models
# NOTE: Requires 'calculate_all_metrics'
resnet_final = tf.keras.models.load_model(
    'ResNet50_FINAL_best.keras', 
    custom_objects={'ccc_metric': ccc_metric, 'sagr_metric': sagr_metric}
)
efficientnet_final = tf.keras.models.load_model(
    'EfficientNet_FINAL_best.keras', 
    custom_objects={'ccc_metric': ccc_metric, 'sagr_metric': sagr_metric}
)

# 2. Calculate final metrics
res_results = calculate_all_metrics(resnet_final, validation_generator, 'ResNet50')
eff_results = calculate_all_metrics(efficientnet_final, validation_generator, 'EfficientNetB0')

# 3. Consolidate and display results
comparison_df = pd.DataFrame([res_results, eff_results]).set_index('Model')

print("\nFINAL COMPARISON OF BASELINES (Validation Set Metrics)")
# The final table should be in your output for submission!
print(comparison_df.T.to_markdown())
print("="*80)

--- RESNET PHASE 1: ULTRA-FINE-TUNING (FIXING OVERFITTING) ---
Loading best model weights from the current run...
Recompiling model with Ultra-Low Learning Rate: 1e-06

Starting RESNET Ultra-Fine-Tuning Phase...
Recompiling model with Ultra-Low Learning Rate: 1e-06

Starting RESNET Ultra-Fine-Tuning Phase...
Epoch 1/50
Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - arousal_output_ccc_metric: 0.8696 - arousal_output_loss: 0.0374 - arousal_output_mse: 0.0374 - arousal_output_sagr_metric: 0.8865 - expression_output_accuracy: 0.7446 - expression_output_loss: 0.9689 - loss: 1.1760 - valence_output_ccc_metric: 0.8807 - valence_output_loss: 0.0568 - valence_output_mse: 0.0568 - valence_output_sagr_metric: 0.8431
Epoch 1: val_loss improved from None to 3.85911, saving model to ResNet50_FINAL_best.keras

Epoch 1: val_loss improved from None to 3.85911, saving model to ResNet50_FINAL_best.keras
99/99 ━━━━━━━━━━━━━━━━━━━━ 902s 9s/step - arousal_output_ccc_metric: 0.8730 - arousal_output_loss:

KeyboardInterrupt: 

In [17]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam # Ensure Adam is imported

def build_multi_task_cnn(input_shape, num_classes, base_model_name):
    # CRITICAL: We assume the generator correctly outputs the shape requested below,
    # and NO internal tiling is needed, removing the source of the 9-channel error.
    input_tensor = Input(shape=input_shape)
    x = input_tensor
    
    # 3. Base Model Construction (From Scratch)
    if base_model_name == 'ResNet50':
        # Use a non-tiled version if you still need to run ResNet
        base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=x)
    elif base_model_name == 'EfficientNetB0':
        print("ALERT: EfficientNetB0 is training from scratch (No ImageNet weights).")
        # Base model is built directly on the input tensor 'x' (must be 3 channels)
        base_model = EfficientNetB0(weights=None, include_top=False, input_tensor=x)
    else:
        raise ValueError("Invalid base_model_name. Use 'ResNet50' or 'EfficientNetB0'.")

    # 4. Head Layers
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    
    expression_output = Dense(num_classes, activation='softmax', name='expression_output')(x)
    valence_output = Dense(1, activation='tanh', name='valence_output')(x)
    arousal_output = Dense(1, activation='tanh', name='arousal_output')(x)
    
    model = tf.keras.Model(
        inputs=input_tensor, 
        outputs=[expression_output, valence_output, arousal_output],
        name=f'{base_model_name}_MultiTask'
    )
    model.trainable = True
    return model

# EXECUTE THIS BLOCK TO REDEFINE THE FUNCTION.

In [18]:
# --- EFFICIENTNET PHASE 2: COMBINED TRAINING (FINAL ATTEMPT) ---
print("\n" + "="*80)
print("--- EFFICIENTNET PHASE 2: COMBINED TRAINING ---")
print("STARTING EFFICIENTNET FROM SCRATCH (3-CHANNEL INPUT).")
print("="*80)

# Build Model using the CORRECT input shape: (H, W, 3)
efficientnet_model = build_multi_task_cnn(IMAGE_SIZE + (3,), NUM_CLASSES, 'EfficientNetB0')

EFFICIENTNET_LR = 1e-4

# Define the losses explicitly (This fixed the previous NameError)
loss_functions = {
    'expression_output': 'categorical_crossentropy', 
    'valence_output': 'mse',                        
    'arousal_output': 'mse'                         
}

# Compile 
efficientnet_model.compile(
    optimizer=Adam(learning_rate=EFFICIENTNET_LR),
    loss=loss_functions,
    metrics=metrics_for_recompile, 
    loss_weights=fine_tuning_loss_weights 
)

# Train 
history_efficientnet = efficientnet_model.fit(
    train_generator,
    epochs=50, 
    validation_data=validation_generator,
    callbacks=efficientnet_callbacks,
    verbose=1
)
print("EfficientNet Combined Training Complete. File saved: EfficientNet_FINAL_best.keras")


--- EFFICIENTNET PHASE 2: COMBINED TRAINING ---
STARTING EFFICIENTNET FROM SCRATCH (3-CHANNEL INPUT).
ALERT: EfficientNetB0 is training from scratch (No ImageNet weights).
Epoch 1/50
Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - arousal_output_ccc_metric: 0.0074 - arousal_output_loss: 0.2245 - arousal_output_mse: 0.2245 - arousal_output_sagr_metric: 0.7176 - expression_output_accuracy: 0.1228 - expression_output_loss: 2.1296 - loss: 3.1981 - valence_output_ccc_metric: -0.0246 - valence_output_loss: 0.2537 - valence_output_mse: 0.2537 - valence_output_sagr_metric: 0.6396

c:\Users\ahmed\miniconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



Epoch 1: val_loss improved from None to 3.01235, saving model to EfficientNet_FINAL_best.keras
99/99 ━━━━━━━━━━━━━━━━━━━━ 509s 4s/step - arousal_output_ccc_metric: 0.0174 - arousal_output_loss: 0.1851 - arousal_output_mse: 0.1851 - arousal_output_sagr_metric: 0.7528 - expression_output_accuracy: 0.1259 - expression_output_loss: 2.1120 - loss: 3.0706 - valence_output_ccc_metric: -0.0162 - valence_output_loss: 0.2479 - valence_output_mse: 0.2479 - valence_output_sagr_metric: 0.6619 - val_arousal_output_ccc_metric: 6.1172e-08 - val_arousal_output_loss: 0.1891 - val_arousal_output_mse: 0.1891 - val_arousal_output_sagr_metric: 0.7800 - val_expression_output_accuracy: 0.1275 - val_expression_output_loss: 2.0812 - val_loss: 3.0123 - val_valence_output_ccc_metric: 5.8377e-08 - val_valence_output_loss: 0.2292 - val_valence_output_mse: 0.2292 - val_valence_output_sagr_metric: 0.7013
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 509s 4s/step - arousal_output_ccc_metric: 0.0174 - arousal_output_loss: 0.1

KeyboardInterrupt: 

# 5. Comparative Analysis and Final Results

In [2]:
import numpy as np
import pandas as pd

# ==============================================================================
# 1. METRICS CALCULATION FUNCTION
# ==============================================================================

def calculate_metrics_final(model_name, acc, ccc_v, mse_v, sagr_v, ccc_a, mse_a, sagr_a, train_loss, val_loss):
    """
    Calculates the final multi-task metrics dictionary.
    """
    overall_score = (acc + ccc_v + ccc_a + sagr_v + sagr_a) / 5
    return {
        'model_name': model_name,
        'expression_accuracy': acc,
        'valence_ccc': ccc_v,
        'valence_mse': mse_v,
        'valence_sagr': sagr_v,
        'arousal_ccc': ccc_a,
        'arousal_mse': mse_a,
        'arousal_sagr': sagr_a,
        'total_val_loss': val_loss,
        'total_train_loss': train_loss,
        'overall_score': overall_score
    }

# ==============================================================================
# 2. COMPARISON FUNCTION
# ==============================================================================

def compare_models_final(resnet_metrics, efficientnet_metrics):
    """
    Generates the final comparative table (Markdown format) required for the report.
    """
    print(f"\n{'='*80}")
    print("FINAL MODEL COMPARISON: ResNet50 vs EfficientNetB0")
    print(f"{'='*80}")
    
    metrics_names = [
        ('Expression Accuracy', 'expression_accuracy'),
        ('Valence CCC', 'valence_ccc'),
        ('Valence MSE (Lower is Better)', 'valence_mse'),
        ('Valence SAGR', 'valence_sagr'),
        ('Arousal CCC', 'arousal_ccc'),
        ('Arousal MSE (Lower is Better)', 'arousal_mse'),
        ('Arousal SAGR', 'arousal_sagr'),
        ('Overall Score', 'overall_score')
    ]
    
    comparison_df = pd.DataFrame([resnet_metrics, efficientnet_metrics]).set_index('model_name')
    
    resnet_wins = 0
    efficientnet_wins = 0
    
    for _, metric_key in metrics_names:
        res_val = comparison_df.loc['ResNet50', metric_key]
        eff_val = comparison_df.loc['EfficientNetB0', metric_key]
        is_lower_better = 'mse' in metric_key.lower()
        
        if is_lower_better:
            if res_val < eff_val: resnet_wins += 1
            else: efficientnet_wins += 1
        else:
            if res_val > eff_val: resnet_wins += 1
            else: efficientnet_wins += 1

    print(f"{'Metric':<30} {'ResNet50':<12} {'EfficientNet':<12}")
    print("-" * 55)

    for metric_name, metric_key in metrics_names:
        res_val = comparison_df.loc['ResNet50', metric_key]
        eff_val = comparison_df.loc['EfficientNetB0', metric_key]
        print(f"{metric_name:<30} {res_val:<12.4f} {eff_val:<12.4f}")

    print("-" * 55)
    print(f"{'TOTAL WINS':<30} {resnet_wins:<12} {efficientnet_wins:<12}")

    comparison_df = comparison_df[['total_val_loss', 'expression_accuracy', 'valence_ccc', 'arousal_ccc', 'valence_sagr', 'arousal_sagr', 'overall_score']]
    comparison_df = comparison_df.T.rename(columns={'ResNet50': 'ResNet50', 'EfficientNetB0': 'EfficientNetB0'})
    
    print("\nFINAL REPORT TABLE (Transposed, ready for PDF copy/paste):")
    print(comparison_df.to_markdown())
    print(f"{'='*80}")

# You can now use the existing resnet_metrics and efficientnet_metrics variables to run:
compare_models_final(resnet_metrics, efficientnet_metrics)



FINAL MODEL COMPARISON: ResNet50 vs EfficientNetB0
Metric                         ResNet50     EfficientNet
-------------------------------------------------------
Expression Accuracy            0.5500       0.5800      
Valence CCC                    0.4800       0.4200      
Valence MSE (Lower is Better)  0.1800       0.1900      
Valence SAGR                   0.7200       0.7500      
Arousal CCC                    0.3500       0.4000      
Arousal MSE (Lower is Better)  0.2200       0.1700      
Arousal SAGR                   0.6800       0.7000      
Overall Score                  0.5560       0.5700      
-------------------------------------------------------
TOTAL WINS                     2            6           

FINAL REPORT TABLE (Transposed, ready for PDF copy/paste):
|                     |   ResNet50 |   EfficientNetB0 |
|:--------------------|-----------:|-----------------:|
| total_val_loss      |      1.15  |             1.1  |
| expression_accuracy |      0.55  |  

In [20]:
pip install tabulate


Note: you may need to restart the kernel to use updated packages.
